# Lab 0 Task 0.1 — CIFAR-10 CNN Experiments
This notebook implements a simple CNN for CIFAR-10 classification and compares different optimizers and activation functions as required for Task 0.1 of Lab 0.

## Setup & Imports

In [1]:
!nvidia-smi # Take a look at the GPU

Wed Mar 25 16:24:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:1A:00.0 Off |                  N/A |
| 29%   27C    P8              1W /  250W |       1MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%pip install numpy
%pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import warnings
import wandb
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

2.11.0+cu130
True
NVIDIA GeForce RTX 2080 Ti
Using device: cuda


## Data Loading

In [4]:
BATCH_SIZE = 256 # Same batch size throughout notebook

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_set = CIFAR10(root="../data", train=True,  download=True, transform=transform)
test_set  = CIFAR10(root="../data", train=False, download=True, transform=transform)

classes: list[str] = train_set.classes
print("Classes:", classes)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=8, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## Model Definition

The `activation` argument lets us swap between `LeakyReLU` and `Tanh` without rewriting the class.

In [5]:
class SimpleCNN(nn.Module):
    """
    A simple CNN for CIFAR-10 classification.

    Architecture:
        Conv Block 1 : Conv2d(3  → 32)  → act → MaxPool2d  (32×32 → 16×16)
        Conv Block 2 : Conv2d(32 → 64)  → act → MaxPool2d  (16×16 →  8×8)
        Conv Block 3 : Conv2d(64 → 128) → act → MaxPool2d  ( 8×8  →  4×4)
        FC 1         : Linear(128*4*4 → 256) → act
        FC 2         : Linear(256 → num_classes)
    """

    def __init__(self, num_classes: int = 10, activation: nn.Module = nn.LeakyReLU()) -> None:
        super().__init__()

        self.conv1 = nn.Conv2d(3,  32,  kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.act  = activation
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(self.act(self.conv1(x)))
        x = self.pool(self.act(self.conv2(x)))
        x = self.pool(self.act(self.conv3(x)))
        x = torch.flatten(x, start_dim=1)
        x = self.act(self.fc1(x))
        return self.fc2(x)

## Training & Evaluation Helpers

`train_one_epoch` and `validate` each handle a single pass through their respective loader.  
`fit` orchestrates the loop, tracks history, and restores the best checkpoint.

In [6]:
def train(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
) -> tuple[float, float]:
    """Run one full pass over `loader` in training mode."""
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        correct      += (outputs.argmax(dim=1) == labels).sum().item()
        total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def validate(
    model: nn.Module,
    loader: DataLoader,
) -> tuple[float, float]:
    """Run one full pass over `loader` in evaluation mode."""
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            correct      += (outputs.argmax(dim=1) == labels).sum().item()
            total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def fit(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    train_loader: DataLoader,
    val_loader: DataLoader,
    run_name: str,
    tags: list[str],
    config: dict,
    num_epochs: int = 10,
) -> dict[str, list[float]]:
    """Train `model` for `num_epochs`, validating after every epoch.

    Initialises and closes a wandb run for the duration of training.
    Saves the weights that achieved the best validation accuracy and
    restores them at the end.

    Returns
    -------
    history : dict with keys 'train_loss', 'val_loss', 'train_acc', 'val_acc'
    """
    wandb.finish()  # close any previously open run
    wandb.init(
        entity="d7047e-group12",
        project="Lab0",
        name=run_name,
        tags=tags,
        config=config,
    )

    best_val_acc = 0.0
    best_state   = None
    history: dict[str, list[float]] = {
        "train_loss": [], "val_loss": [],
        "train_acc":  [], "val_acc":  [],
    }

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train(model, train_loader, optimizer)
        val_loss,   val_acc   = validate(model, val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:>{len(str(num_epochs))}}/{num_epochs} | "
            f"train loss {train_loss:.4f}, train acc {train_acc:.2f}% | "
            f"val loss {val_loss:.4f}, val acc {val_acc:.2f}%"
        )

        wandb.log({
            "Training Loss": train_loss,
            "Training Accuracy": train_acc,
            "Validation Loss": val_loss,
            "Validation Accuracy": val_acc
        })

    # Restore best checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\nRestored best weights (val acc {best_val_acc:.2f}%)")

    wandb.finish()
    return history

## Experiment A – SGD + LeakyReLU
*(The original baseline from the script)*

In [7]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-2

criterion = nn.CrossEntropyLoss()

model_a = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_a = optim.SGD(model_a.parameters(), lr=LEARNING_RATE)

print("=== Experiment A: SGD + LeakyReLU ===")
history_a = fit(
    model_a, optimizer_a, train_loader, test_loader,
    tags=["Task 0.1"],
    run_name="SGD + LeakyReLU",
    config={"optimizer": "SGD", "lr": LEARNING_RATE, "activation": "LeakyReLU",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
    num_epochs = NUM_EPOCHS
)

=== Experiment A: SGD + LeakyReLU ===


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: oscar-engelmark (d7047e-group12) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch  1/50 | train loss 2.2986, train acc 11.89% | val loss 2.2934, val acc 13.14%
Epoch  2/50 | train loss 2.2839, train acc 16.65% | val loss 2.2677, val acc 21.61%
Epoch  3/50 | train loss 2.2189, train acc 22.89% | val loss 2.1403, val acc 25.81%
Epoch  4/50 | train loss 2.0795, train acc 26.48% | val loss 2.0307, val acc 27.58%
Epoch  5/50 | train loss 1.9915, train acc 29.46% | val loss 1.9433, val acc 31.41%
Epoch  6/50 | train loss 1.9041, train acc 32.30% | val loss 1.8754, val acc 32.56%
Epoch  7/50 | train loss 1.8278, train acc 34.75% | val loss 1.7871, val acc 36.37%
Epoch  8/50 | train loss 1.7624, train acc 36.89% | val loss 1.7649, val acc 36.09%
Epoch  9/50 | train loss 1.7131, train acc 38.40% | val loss 1.7339, val acc 36.62%
Epoch 10/50 | train loss 1.6536, train acc 40.26% | val loss 1.5962, val acc 42.28%
Epoch 11/50 | train loss 1.6087, train acc 42.34% | val loss 1.5570, val acc 43.27%
Epoch 12/50 | train loss 1.5601, train acc 43.65% | val loss 1.5394, val acc

Training Accuracy,▁▂▂▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇███████
Training Loss,███▇▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
Validation Accuracy,▁▂▃▃▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▇▇▆▇▇▇▇▇▇▇▇▇████████▇
Validation Loss,██▇▇▆▅▅▅▄▄▄▃▃▃▃▃▃▃▃▂▂▂▃▂▂▂▂▂▂▃▂▁▁▁▁▁▁▁▁▂
Training Accuracy,69.216
Training Loss,0.89192
Validation Accuracy,58.93
Validation Loss,1.1593


## Experiment B – Adam + LeakyReLU
*(Task: swap SGD for Adam and report accuracy)*

In [8]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3

model_b     = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_b = optim.Adam(model_b.parameters(), lr=LEARNING_RATE)

print("=== Experiment B: Adam + LeakyReLU ===")
history_b = fit(
    model_b, optimizer_b, train_loader, test_loader,
    tags=["Task 0.1"],
    run_name="Adam + LeakyReLU",
    config={"optimizer": "Adam", "lr": LEARNING_RATE, "activation": "LeakyReLU",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
    num_epochs = NUM_EPOCHS
)

=== Experiment B: Adam + LeakyReLU ===


Epoch  1/50 | train loss 1.5095, train acc 45.29% | val loss 1.2840, val acc 53.92%
Epoch  2/50 | train loss 1.1215, train acc 60.23% | val loss 1.0959, val acc 60.11%
Epoch  3/50 | train loss 0.9320, train acc 66.91% | val loss 0.9661, val acc 66.37%
Epoch  4/50 | train loss 0.8050, train acc 71.97% | val loss 0.8349, val acc 70.53%
Epoch  5/50 | train loss 0.7001, train acc 75.59% | val loss 0.7878, val acc 72.62%
Epoch  6/50 | train loss 0.6095, train acc 78.69% | val loss 0.7702, val acc 73.88%
Epoch  7/50 | train loss 0.5347, train acc 81.46% | val loss 0.7846, val acc 74.04%
Epoch  8/50 | train loss 0.4618, train acc 83.96% | val loss 0.7822, val acc 73.83%
Epoch  9/50 | train loss 0.3897, train acc 86.55% | val loss 0.8041, val acc 73.73%
Epoch 10/50 | train loss 0.3180, train acc 89.04% | val loss 0.8384, val acc 74.31%
Epoch 11/50 | train loss 0.2628, train acc 91.02% | val loss 0.8815, val acc 74.69%
Epoch 12/50 | train loss 0.1950, train acc 93.51% | val loss 0.9411, val acc

Training Accuracy,▁▃▄▄▅▆▆▆▇▇██████████████████████████████
Training Loss,█▆▅▅▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation Accuracy,▁▃▅▆▇▇▇▇▇████████████████▇██████▇███▇███
Validation Loss,▃▂▁▁▁▁▁▁▁▂▂▃▃▄▄▄▅▅▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▆▇▇▇▇█
Training Accuracy,99.416
Training Loss,0.01704
Validation Accuracy,74.63
Validation Loss,2.41237


## Experiment C – Adam + Tanh
*(Task: swap LeakyReLU for Tanh and report accuracy)*

In [9]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3

model_c     = SimpleCNN(num_classes=len(classes), activation=nn.Tanh()).to(device)
optimizer_c = optim.Adam(model_c.parameters(), lr=LEARNING_RATE)

print("=== Experiment C: Adam + Tanh ===")
history_c = fit(
    model_c, optimizer_c, train_loader, test_loader,
    tags=["Task 0.1"],
    run_name="Adam + Tanh",
    config={"optimizer": "Adam", "lr": LEARNING_RATE, "activation": "Tanh",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
    num_epochs = NUM_EPOCHS
)

=== Experiment C: Adam + Tanh ===


Epoch  1/50 | train loss 1.4155, train acc 49.36% | val loss 1.1317, val acc 59.46%
Epoch  2/50 | train loss 1.0170, train acc 64.05% | val loss 0.9483, val acc 66.11%
Epoch  3/50 | train loss 0.8464, train acc 70.46% | val loss 0.8707, val acc 69.36%
Epoch  4/50 | train loss 0.7247, train acc 74.89% | val loss 0.8225, val acc 71.26%
Epoch  5/50 | train loss 0.6184, train acc 78.65% | val loss 0.8060, val acc 72.28%
Epoch  6/50 | train loss 0.5268, train acc 82.17% | val loss 0.7617, val acc 73.90%
Epoch  7/50 | train loss 0.4392, train acc 85.30% | val loss 0.7703, val acc 73.86%
Epoch  8/50 | train loss 0.3525, train acc 88.67% | val loss 0.8015, val acc 72.95%
Epoch  9/50 | train loss 0.2693, train acc 91.87% | val loss 0.7940, val acc 74.59%
Epoch 10/50 | train loss 0.1982, train acc 94.65% | val loss 0.8436, val acc 73.99%
Epoch 11/50 | train loss 0.1370, train acc 96.89% | val loss 0.8694, val acc 74.25%
Epoch 12/50 | train loss 0.0914, train acc 98.41% | val loss 0.8866, val acc

Training Accuracy,▁▃▄▅▅▆▆▇▇███████████████████████████████
Training Loss,█▆▅▅▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation Accuracy,▁▄▅▆▇▇▇█▇▇██████████████████████████████
Validation Loss,▄▃▂▁▁▁▁▁▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇██
Training Accuracy,100
Training Loss,0.00011
Validation Accuracy,75.43
Validation Loss,1.50657


In [10]:
wandb.finish()